# 01b · Train / validation split by time

The train folder covers 6 days (Nov 9–14). We hold out the **last day** for validation:

- **Train:** Nov 9–13. The model learns from these days.
- **Validation:** Nov 14. We check the model here while building it.
- **Test:** the locked `MINDsmall_test` folder (Nov 15). Only for final results.

Splitting by time is like a practice exam: study the past, get tested on a day you haven't seen. A random split would let the model peek at the future and look better than it really is.

All loading, bucketing and splitting code lives in `ownfeed/data.py` so every later notebook uses the same version.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from ownfeed.data import BUCKETS, explode_impressions, load_behaviors, load_news, split_by_time

In [ ]:
news = load_news()
behaviors = load_behaviors()
train, val = split_by_time(behaviors)

pd.DataFrame({
    "impressions": [len(train), len(val)],
    "share_%": [round(100 * len(train) / len(behaviors), 1), round(100 * len(val) / len(behaviors), 1)],
    "users": [train.user_id.nunique(), val.user_id.nunique()],
    "first": [train.time.min(), val.time.min()],
    "last": [train.time.max(), val.time.max()],
}, index=["train", "validation"])

## Check 1: Does the click history leak the future?

Each impression comes with the user's `history` (articles they clicked before). If that history included clicks from Nov 14, training would secretly see validation behavior.

In MIND, history is collected **before** the 6-day log window, so it should be identical across all of a user's impressions. Let's confirm.

In [ ]:
users_with_changing_history = (behaviors.groupby("user_id").history.nunique() > 1).sum()
print(f"Users whose history changes between impressions: {users_with_changing_history}")

## Check 2: How many validation users are new?

A "new" user never appears in Nov 9–13. If they have a click history, the user tower can still describe them from it.

In [ ]:
new_users = set(val.user_id) - set(train.user_id)
new_user_rows = val[val.user_id.isin(new_users)]
print(f"New users: {len(new_users):,} of {val.user_id.nunique():,} validation users ({100 * len(new_users) / val.user_id.nunique():.0f}%)")
print(f"Their share of validation impressions: {100 * len(new_user_rows) / len(val):.0f}%")
print(f"New users with an empty history: {(new_user_rows.drop_duplicates('user_id').history == '').sum():,}")

## Check 3: How many validation articles are brand new?

News is fresh every day. An article is "new" if it never appears in training, neither shown on Nov 9–13 nor in anyone's history.

In [ ]:
train_shown = explode_impressions(train, news)
val_shown = explode_impressions(val, news)

seen_in_train = set(train_shown.news_id) | set(" ".join(train.history).split())
val_articles = val_shown.news_id.drop_duplicates()
is_new = ~val_shown.news_id.isin(seen_in_train)

print(f"Validation articles never seen in training: {100 * (~val_articles.isin(seen_in_train)).mean():.0f}%")
print(f"Share of validation clicks that go to those new articles: {100 * is_new[val_shown.clicked == 1].mean():.0f}%")

## Check 4: Does the bucket mix look similar?

In [ ]:
pd.DataFrame({
    "train_click_share_%": train_shown[train_shown.clicked == 1].bucket.value_counts(normalize=True),
    "val_click_share_%": val_shown[val_shown.clicked == 1].bucket.value_counts(normalize=True),
}).reindex(BUCKETS).mul(100).round(1)

## Takeaways

- **The split:** train = Nov 9–13 (~127K impressions, 81%), validation = Nov 14 (~30K impressions, 19%).
- **No leakage from history.** Every user's history is the same across all their impressions, so it was collected before the log window and can safely be used as input.
- **About 1 in 5 validation users are new**, but they still come with a click history. The user tower should build a user from their history, not from a memorized user ID.
- **About half of validation articles are brand new, and they get roughly a third of the clicks.** This confirms the item tower must read title, abstract and category. A model that only memorizes article IDs would miss a third of what people click.
- **The bucket mix on Nov 14 is a bit different**: more Entertain, less Sports. That matches the day-to-day wobble from the EDA. Fine for validation, but small differences between models may be noise.